# 05 — Flux engine + corpus store; κ on the training grid

The instrument node: a compiled flux engine (ADR 0004) that computes gross
per-column rates f⁺ for every state, persisted as chunked HDF5 under
`data/fluxes/`. From f⁺ the reader materializes f⁻ = f⁺[pair_col], the signed
net flux φ = f⁺ − f⁻, and the **cancellation ratio**

$$\kappa_r = \frac{|f^+_r - f^-_r|}{f^+_r + f^-_r}$$

κ is the load-bearing unknown of the whole project (checklist row 6): if net
flow hides under massive forward/reverse cancellation, Target A's flux
parameterization is in trouble. This notebook shows κ on the **training
grid** — where the answer turns out to be a *vacuous pass*.

Every κ figure declares its convention. **The two κ conventions are never
mixed** (root CLAUDE.md): thresholds are evaluated UNSCREENED; screened κ is
a screening-offset diagnostic only. `nbsupport` enforces this — a mismatched
declaration raises.

Exploratory only — citable values are the RESULTS.md 2026-07-10/11 rows.

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import matplotlib.pyplot as plt
import numpy as np

import nbsupport as nbs
from gnn_nucleo.killtest.distributions import kappa_summary

nbs.style()
QUICK = nbs.QUICK
NETS = ["mesa_80"] if QUICK else ["mesa_80", "mesa_151"]
MAX_CHUNKS = 1 if QUICK else None

In [ ]:
nbs.provenance_header(
    "05",
    "Flux engine + corpus store; training-grid κ",
    nbs.status_of([6]),
    results_rows=[
        "2026-07-10: pf gate implemented (DerivedRate(use_pf=True) compiled in, PfGateError otherwise); κ at NSE unscreened median 2.6e-12 (floor eliminated)",
        "2026-07-10: on the TRAINING DISTRIBUTION κ ≈ 1 everywhere — {κ>0.1} covers 97–100 % of carrying pairs (random compositions are nowhere near flux balance)",
        "2026-07-11/12: training-grid picture confirmed at full corpus scale (1,034,704 in-strata states/net; frac κ>0.1 = 0.985–0.999 / 0.974–0.997, mesa_80/151)",
        "2026-07-10: label-screening config adds a REAL ~7e-2 κ offset at NSE (κ ≈ |Δln scor|) — a config property, not a pf artifact",
    ],
    data=[
        "data/fluxes/{net}/subsample-unscreened, subsample (FluxStore; hashes in RESULTS.md)",
    ],
    scripts=["scripts/step5_run_fluxes.py", "scripts/run_killtest.py --training-grid"],
)

## The store

Chunked HDF5, 16,384 states per chunk. Only f⁺ is persisted — f⁻/φ/κ are
exactly recoverable through `pair_col`, which halves the disk. In QUICK mode
we read a single chunk (~2 s); the full-corpus runs are 64 chunks/4.9 GB per
network and are what the RESULTS.md rows were measured on.

In [ ]:
for net in NETS:
    st = nbs.flux_store(net, "subsample-unscreened")
    attrs = list(nbs.iter_chunk_attrs(st))
    print(f"{net}/subsample-unscreened: {len(attrs)} chunks | "
          f"screening={attrs[0].get('screening')!r} | "
          f"engine commit {attrs[0].get('git_commit')}")
    st_s = nbs.flux_store(net, "subsample")
    a_s = list(nbs.iter_chunk_attrs(st_s))
    print(f"{net}/subsample            : {len(a_s)} chunks | "
          f"screening={a_s[0].get('screening')!r}")

## Figure 1 — κ on the training grid (UNSCREENED)

Per-T9-stratum distribution of log₁₀ κ over carrying strong pairs (pairs with
f⁺ above the per-stratum median of positive f⁺ — the Step-5 convention).

In [ ]:
hists = {}
for net in NETS:
    st = nbs.flux_store(net, "subsample-unscreened")
    hists[net], bins, nread = nbs.kappa_hist_by_stratum(
        st, convention="UNSCREENED", max_chunks=MAX_CHUNKS
    )
    print(f"{net}: {nread} chunk(s) read, {hists[net].sum():,} κ samples")

centers = 0.5 * (bins[:-1] + bins[1:])
net = NETS[0]
h = hists[net]

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=True, sharey=True)
for i, (ax, lab) in enumerate(zip(axes.ravel(), nbs.T9_LABELS)):
    tot = h[i].sum()
    if tot == 0:
        ax.set_visible(False)
        continue
    ax.fill_between(centers, h[i] / tot, step="mid", alpha=0.75, color="#0072B2")
    ax.axvline(-1, color="#D55E00", ls="--", lw=1.2)
    frac = h[i][centers > -1].sum() / tot
    ax.set_title(f"T₉ {lab} — frac κ>0.1 = {frac:.3f}", fontsize=9)
    ax.set_xlim(-16, 0)
for ax in axes[-1]:
    ax.set_xlabel("log₁₀ κ")
for ax in axes[:, 0]:
    ax.set_ylabel("fraction of carrying pairs")
fig.suptitle(f"{net} — κ on the TRAINING GRID · UNSCREENED convention "
             f"(dashed = the 0.1 active-set gate)", y=1.01)
if QUICK:
    nbs.quick_banner(fig)
nbs.caption(
    fig,
    "κ piles up at O(1) in every stratum: the Sobol training states are random compositions, "
    "nowhere near flux balance, so essentially every reaction carries net flow. The active-set "
    "criterion 'passes' here — but VACUOUSLY: it is a statement about the sampling design, not "
    "about silicon burning. This is exactly why the verdict needed the relaxed manifold "
    "(notebook 10), where the picture is structured.",
    results=[
        "RESULTS.md 2026-07-11/12 training-grid rows (frac κ>0.1 = 0.985–0.999 / 0.974–0.997 at 1,034,704 in-strata states/net)",
        "RESULTS.md 2026-07-10 training-distribution κ row (97–100 % of carrying pairs)",
    ],
    scripts=["scripts/run_killtest.py --training-grid", "scripts/step5_run_fluxes.py"],
)

## Figure 2 — the vacuous pass, summarized

`killtest.distributions.kappa_summary` is the measurement function the
kill-test script calls (it refuses screened runs by default). Below: its
per-stratum frac κ>0.1 and frac κ<1e-3 on the same data.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.4))
width = 0.8 / len(NETS)
for j, net in enumerate(NETS):
    st = nbs.flux_store(net, "subsample-unscreened")
    rows = kappa_summary(st) if not QUICK else None
    if rows is None:
        # QUICK: derive the same statistic from the single-chunk histogram
        h = hists[net]
        labs, fr = [], []
        for i, lab in enumerate(nbs.T9_LABELS):
            tot = h[i].sum()
            if tot:
                labs.append(lab)
                fr.append(h[i][centers > -1].sum() / tot)
    else:
        labs = [r["stratum"] for r in rows]
        fr = [r["frac_gt_0p1"] for r in rows]
    x = np.arange(len(labs))
    ax.bar(x + (j - (len(NETS) - 1) / 2) * width, fr, width, label=net)
ax.axhline(0.95, color="#D55E00", ls="--", label="Target-A viability line (≥95 %)")
ax.set_xticks(np.arange(len(labs)), labs)
ax.set_ylim(0.9, 1.005)
ax.set_xlabel("T₉ stratum")
ax.set_ylabel("frac of carrying pairs with κ > 0.1")
ax.set_title("Training grid: the active set is ~everything (UNSCREENED)")
ax.legend(fontsize=8, loc="lower left")
if QUICK:
    nbs.quick_banner(fig)
nbs.caption(
    fig,
    "Every stratum clears the ≥95 % line with room to spare, and κ-balanced (κ<1e-3) pairs are "
    "measured at 0.000 — no equilibrated sector exists on this distribution at all. A design "
    "validated only here would be untested against the physics it will actually meet.",
    results=["RESULTS.md 2026-07-11/12 training-grid rows (frac κ<1e-3 = 0.000)"],
    scripts=["scripts/run_killtest.py --training-grid"],
)

## Figure 3 — SCREENED κ: the screening-offset diagnostic

**Separate figure, separate convention, on purpose.** The label config
(`chugunov_2007`) applies screening per reaction, so a screened capture pairs
with an unscreened photodissociation and κ acquires a floor of |Δln scor| ≈
7e-2 at NSE. That offset is **real** — a property of the label
configuration, not a partition-function artifact (contrast notebook 04's
spurious pf-free floor, which was a bug). It is why equilibrium detection
must use unscreened κ: a 7e-2 floor would swamp the 1e-3 balance criterion.

In [ ]:
net = NETS[0]
h_s, _, n_s = nbs.kappa_hist_by_stratum(
    nbs.flux_store(net, "subsample"),
    convention="SCREENED",
    allow_screened=True,  # the ONE legitimate site: a screening-offset diagnostic
    max_chunks=MAX_CHUNKS,
)

fig, ax = plt.subplots(figsize=(10, 4.6))
i_hot = len(nbs.T9_LABELS) - 1
for hh, label, c in [
    (hists[net][i_hot], "UNSCREENED — separate run, shown for scale", "#0072B2"),
    (h_s[i_hot], "SCREENED (chugunov_2007) — label config", "#D55E00"),
]:
    tot = hh.sum()
    if tot:
        ax.fill_between(centers, hh / tot, step="mid", alpha=0.5, color=c, label=label)
ax.axvline(np.log10(7e-2), color="#333333", ls=":", lw=1.4)
ax.text(np.log10(7e-2) + 0.15, ax.get_ylim()[1] * 0.75,
        "κ ≈ |Δln scor| ≈ 7e-2 at NSE\n(REAL config offset, RESULTS 2026-07-10)",
        fontsize=8)
ax.set_xlim(-16, 0)
ax.set_xlabel("log₁₀ κ")
ax.set_ylabel("fraction of carrying pairs")
ax.set_title(f"{net}, T₉ {nbs.T9_LABELS[i_hot]} — screening-offset diagnostic ONLY\n"
             "(κ thresholds are NEVER evaluated on the screened run)")
ax.legend(fontsize=8, loc="upper left")
if QUICK:
    nbs.quick_banner(fig)
nbs.caption(
    fig,
    "Shown together ONLY to make the offset visible and label it; every κ THRESHOLD in this "
    "project (the 0.1 active-set gate, the 1e-3 balance criterion) is evaluated on the unscreened "
    "run. On the training grid both conventions sit at κ ≈ 1 anyway — the offset matters at NSE, "
    "where a 7e-2 screened floor would masquerade as 'not equilibrated'. nbsupport refuses a "
    "convention declaration that disagrees with the run's own attrs.",
    results=[
        "RESULTS.md 2026-07-10 screened-κ offset rows (κ ≈ |Δln scor| ≈ 7e-2 at NSE; docs/rate-crosscheck.md screened-κ caveat)",
    ],
    scripts=["scripts/step5_run_fluxes.py"],
)

## TODO (stub)

- **Full-corpus confirmation** (64 chunks/net, `run_id="full"`, 4.9 GB):
  measured at 1,034,704 in-strata states/net (RESULTS.md 2026-07-11/12).
  NOTE the `full` runs are **screened** (chugunov_2007) — no unscreened twin
  exists at corpus scale — so re-plotting there needs
  `convention="SCREENED", allow_screened=True` and is a screened-κ reading,
  not a threshold evaluation. That is what the Task-2A corpus row did, and
  it is only admissible because on the training distribution κ ≈ 1 dwarfs
  the 7e-2 screening offset (RESULTS.md 2026-07-11/12). `NB_QUICK=0` here
  stays on the unscreened subsample and simply reads both networks.

## What this notebook does NOT show

- κ on the **relaxed manifold** — where the distribution is structured and
  the verdict actually lives: notebook 10.
- The pf gate's enforcement (`PfGateError`) and the raw-v-flag floor it
  prevents: notebook 04.
- Bridge/inter-group structure derived from these fluxes: notebook 06.